# Titanic SVM Classification Assignment
This notebook follows the assignment requirements: EDA, preprocessing, feature engineering, SVM models (Linear, Polynomial, RBF), scaling comparison, and analysis.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,classification_report,ConfusionMatrixDisplay


ModuleNotFoundError: No module named 'pandas'

In [ ]:
df=pd.read_csv(r""C:\Users\adipi\Downloads\tested.csv"")
df.head()

## 1. Exploratory Data Analysis

In [ ]:
print(df.shape)
display(df.head(10))
display(df.describe(include='all'))
print(df.dtypes)
print(df.isnull().sum())

num=df.select_dtypes(include=np.number).columns
cat=df.select_dtypes(exclude=np.number).columns

plt.figure(figsize=(8,5))
sns.heatmap(df.isnull(),cbar=False)
plt.title("Missing Values")
plt.show()

if 'Survived' in df.columns:
    sns.countplot(data=df,x='Survived')
    plt.title("Class Distribution")
    plt.show()

plt.figure(figsize=(10,8))
sns.heatmap(df[num].corr(),annot=True,cmap='coolwarm')
plt.show()


## 2. Preprocessing & Feature Engineering

In [ ]:
work=df.copy()

if 'Name' in work.columns:
    work['Title']=work['Name'].str.extract(' ([A-Za-z]+)\.',expand=False)

if set(['SibSp','Parch']).issubset(work.columns):
    work['FamilySize']=work['SibSp']+work['Parch']+1
    work['IsAlone']=(work['FamilySize']==1).astype(int)

if 'Age' in work.columns:
    work['AgeGroup']=pd.cut(work['Age'],bins=[0,12,18,35,60,100],labels=False)

if 'Fare' in work.columns:
    work['FareGroup']=pd.qcut(work['Fare'],4,labels=False,duplicates='drop')

drop_cols=[c for c in ['PassengerId','Ticket','Cabin','Name'] if c in work.columns]
work=work.drop(columns=drop_cols)

target='Survived'
X=work.drop(columns=[target])
y=work[target]

num_cols=X.select_dtypes(include='number').columns
cat_cols=X.select_dtypes(exclude='number').columns

pre=ColumnTransformer([
('num',Pipeline([('imp',SimpleImputer(strategy='median')),
                 ('scaler',StandardScaler())]),num_cols),
('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent'))]),cat_cols)
],remainder='drop')

X=pd.get_dummies(X,dummy_na=True)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


## 3. Feature Set A and B with SVM Kernels

In [ ]:
kernels=['linear','poly','rbf']
for k in kernels:
    model=Pipeline([
        ('imp',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler()),
        ('svm',SVC(kernel=k))
    ])
    model.fit(X_train,y_train)
    tr=accuracy_score(y_train,model.predict(X_train))
    te=accuracy_score(y_test,model.predict(X_test))
    print(f'Kernel: {k}')
    print('Train:',tr,' Test:',te)
    print(classification_report(y_test,model.predict(X_test)))
    ConfusionMatrixDisplay.from_predictions(y_test,model.predict(X_test))
    plt.show()


## 4. Scaling vs No Scaling

In [ ]:
plain=Pipeline([
('imp',SimpleImputer(strategy='median')),
('svm',SVC(kernel='rbf'))
])
plain.fit(X_train,y_train)
print("Without Scaling:",accuracy_score(y_test,plain.predict(X_test)))

scaled=Pipeline([
('imp',SimpleImputer(strategy='median')),
('scaler',StandardScaler()),
('svm',SVC(kernel='rbf'))
])
scaled.fit(X_train,y_train)
print("With Scaling:",accuracy_score(y_test,scaled.predict(X_test)))


## 5. Analysis

- Linear SVM is simple and often generalizes well.
- Polynomial kernel may overfit depending on degree.
- RBF usually provides the best balance for nonlinear decision boundaries.
- Feature engineering (Title, FamilySize, IsAlone, AgeGroup, FareGroup) improves representation.
- Scaling is essential because SVM relies on distance calculations. Without scaling, features with larger numeric ranges dominate the optimization process and performance usually decreases.
